filter function is going to retrive only those item which are meeting a certain condition in an iterable 

## list(filter(func,iterable))

In [1]:
def even_function(value):
    if value%2 ==0:
        return True
        

In [2]:
iterable_lsit = [1,2,3,4,5,6,7,8,9,10]
print(list(filter(even_function, iterable_lsit  )))

[2, 4, 6, 8, 10]


In [4]:
print(list(filter(lambda x: x % 2 == 0, iterable_lsit)))

[2, 4, 6, 8, 10]


In [5]:
print(list(filter(lambda x:x>5,iterable_lsit)))

[6, 7, 8, 9, 10]


In [8]:
employee  = [{'name':'John', 'age':30, 'qualification':'Bachelors'},
             {'name':'Jane', 'age':25, 'qualification':'Masters'},
             {'name':'Doe', 'age':35, 'qualification':'PhD'}]



In [9]:
def age_greater(employee):
    if employee['age'] > 25:
        return True


In [10]:
print(list(filter(age_greater, employee)))

[{'name': 'John', 'age': 30, 'qualification': 'Bachelors'}, {'name': 'Doe', 'age': 35, 'qualification': 'PhD'}]


In [11]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Set seed for reproducibility
np.random.seed(42)
sample_size = 400

# 2. Define the Target Statistical Properties from the Paper
# Means and Standard Deviations
means = {
    "FCV_19S": 16.09,
    "Spiegel": 17.29,
    "SAS": 52.47,
    "SDS": 65.89,
    "Age": 45.75,
}

stds = {
    "FCV_19S": 6.81,
    "Spiegel": 6.36,
    "SAS": 10.39,
    "SDS": 8.72,
    "Age": 15.04,
}

# Target Pearson Correlation Matrix (Table 5)
# Columns order: FCV_19S, Spiegel, SAS, SDS, Age
# Note: Age correlation wasn't fully detailed in Table 5, so we approximate its slight negative trend.
corr_matrix = np.array(
    [
        [1.00, 0.62, 0.36, 0.56, -0.05],  # FCV-19S
        [0.62, 1.00, 0.47, 0.68, -0.08],  # Spiegel (Insomnia)
        [0.36, 0.47, 1.00, 0.16, -0.10],  # SAS (Anxiety)
        [0.56, 0.68, 0.16, 1.00, -0.02],  # SDS (Depression)
        [-0.05, -0.08, -0.10, -0.02, 1.00],  # Age
    ]
)

# 3. Generate Simulated Continuous Data
# Reconstruct Covariance Matrix from Correlation Matrix: Cov = Diag(Std) * Corr * Diag(Std)
std_vector = np.array([stds[k] for k in means.keys()])
mean_vector = np.array([means[k] for k in means.keys()])
cov_matrix = np.diag(std_vector) @ corr_matrix @ np.diag(std_vector)

simulated_data = np.random.multivariate_normal(
    mean_vector, cov_matrix, size=sample_size
)
df = pd.DataFrame(simulated_data, columns=list(means.keys()))

# 4. Generate Categorical/Demographic Data (Table 1 Distributions)
# Marital Status: 76.3% Married (1), 20.8% Unmarried (0), 3.1% Divorced/Widowed (2)
df["Marital_Status"] = np.random.choice(
    [1, 0, 2], size=sample_size, p=[0.763, 0.208, 0.029]
)

# Occupation: 27% Manual, 38.8% Clerk, 12% Student, 22.3% Retiree/Unemployed
df["Occupation"] = np.random.choice(
    ["Manual", "Clerk", "Student", "Unemployed"],
    size=sample_size,
    p=[0.270, 0.388, 0.120, 0.222],
)

# Clip clinical metrics to their logical boundaries
df["Spiegel"] = df["Spiegel"].clip(0, 42).round()
df["FCV_19S"] = df["FCV_19S"].clip(7, 35).round()
df["SAS"] = df["SAS"].clip(20, 80).round()
df["SDS"] = df["SDS"].clip(20, 80).round()
df["Age"] = df["Age"].clip(18, 80).round()

print("--- Simulated Dataset Created ---")
print(df.head())
print("\n" + "=" * 50 + "\n")


# 5. Perform Pearson Correlation Analysis
print("1. PEARSON CORRELATION MATRIX (Matches Table 5 Patterns)")
correlation_results = df[["FCV_19S", "Spiegel", "SAS", "SDS"]].corr(
    method="pearson"
)
print(correlation_results.round(2))
print("\n" + "=" * 50 + "\n")


# 6. Perform Multiple Linear Regression Analysis
print("2. MULTIPLE LINEAR REGRESSION ANALYSIS (Matches Table 6 Setup)")

# statsmodels handles categorical dummy variables automatically using the 'C()' notation
model_formula = (
    "Spiegel ~ FCV_19S + SAS + SDS + Age + C(Marital_Status) + C(Occupation)"
)
regression_model = smf.ols(formula=model_formula, data=df).fit()

# View the full analysis report
print(regression_model.summary())

ModuleNotFoundError: No module named 'numpy'